# A morning at one charging point

- **Method:** [protocol.ipynb](protocol.ipynb), Sections II–III. Load its exact definitions; no copied implementation.
- **Clock:** minutes since midnight; requests arrive at 08:05, 08:10 and 08:15. Durations reserve **access**, not measured energy or observed use.
- **Outcomes to check:** C receives full service; D is readmitted at a permitted shorter duration; E has feasible service but no economic agreement.
- **Run:** `uv sync --group notebooks`, select that environment, restart and Run All. Root and `notebooks/` working directories are supported.

In [1]:
from fractions import Fraction as F
from pathlib import Path
import portion as P
import interval_logic as il
from protocol import (Booking, Request, Service, standing, affine_policies, decide_booking,
                      independent_feasible_starts, joint_witness, table, display_timeline, export_timelines)
from protocol.companion import load_method_notebook, clock, clock_interval, plot_charging_morning

NOTEBOOK_DIR = Path('notebooks') if Path('notebooks').is_dir() else Path('.')
method = load_method_notebook(NOTEBOOK_DIR / 'protocol.ipynb')

## Inputs

Start windows are **closed**; bookings are half-open. Sampling uses the method's $d/2$ grid.

| Client | Received | Original service | Permitted alternative |
|---|---|---|---|
| C | 08:05 | 60 min; starts 09:00–09:30 | None |
| D | 08:10 | 60 min; starts 10:00–10:15 | Same start window, 30 min |
| E | 08:15 | 60 min; starts 11:30–12:00 | None |

**Declared policies:** $u_o=q-c$, $u_c=v-q$; earliest legal start; target-based acceptance;
halfway concessions; three rounds; legal prices €0–€12.
C/D use $(c,v)=(2,8)$, initial targets $(4,6)$ and minima $(1,1)$.
E uses $(c,v)=(8,6)$, targets/minima $(0,0)$: owner needs $q\geq8$, client needs $q\leq6$.

In [2]:
NAMES = {1: 'A', 2: 'B', 3: 'C', 4: 'D', 5: 'E'}
INITIAL = (Booking(1, P.closedopen(480, 540)), Booking(2, P.closedopen(630, 690)))
PRICES = (F(0), F(12))
ROUND_BUDGET = 3
REQUESTS = (
    dict(received=F(485), request=Request(3, Service(P.closed(540, 570), 60)),
         cost=F(2), value=F(8), targets={'owner': F(4), 'client': F(6)}, minima={'owner': F(1), 'client': F(1)}),
    dict(received=F(490), request=Request(4, Service(P.closed(600, 615), 60), (Service(P.closed(600, 615), 30),)),
         cost=F(2), value=F(8), targets={'owner': F(4), 'client': F(6)}, minima={'owner': F(1), 'client': F(1)}),
    dict(received=F(495), request=Request(5, Service(P.closed(690, 720), 60)),
         cost=F(8), value=F(6), targets={'owner': F(0), 'client': F(0)}, minima={'owner': F(0), 'client': F(0)}),
)

## Three sequential attempts

`attempt` only aggregates the unchanged `admit → generate → bargain → commit` methods.
Each attempt retains its before/after snapshots; the next uses the updated commitments.

In [3]:
def attempt(spec, commitment, bookings, ledger):
    request, now = spec['request'], spec['received']
    admission = method.admit(request, commitment, now)
    assert admission is not None
    offers = method.generate(admission)
    policies = affine_policies({o: spec['cost'] for o in offers}, {o: spec['value'] for o in offers})
    negotiation = method.bargain(offers, policies, spec['targets'], spec['minima'], PRICES, ROUND_BUDGET)
    result = method.commit(admission, offers, negotiation, commitment, ledger)
    after = bookings if negotiation.agreement is None else (*bookings, negotiation.agreement.offer.booking)

    # Independent scheduling check and preservation of all earlier reservations.
    original_starts = independent_feasible_starts(request.service, bookings, request.client, now)
    assert independent_feasible_starts(admission.service, bookings, request.client, now) == admission.feasible_starts
    previous_unchanged = after[:len(bookings)] == bookings and result.ledger[:len(ledger)] == ledger
    jointly_satisfiable = decide_booking(result.commitment, 0).satisfiable
    assert previous_unchanged and jointly_satisfiable
    assert il.satisfies(result.commitment.guarantee, joint_witness(after, ()), 0)
    if negotiation.agreement is None:
        assert result.commitment == commitment and result.ledger == ledger and after == bookings
    else:
        assert result.commitment == commitment.conjoin(negotiation.agreement.offer.contract)
    return dict(**spec, admission=admission, offers=offers, negotiation=negotiation, result=result,
                bookings_before=bookings, bookings_after=after, original_starts=original_starts,
                previous_unchanged=previous_unchanged, jointly_satisfiable=jointly_satisfiable)

In [4]:
commitment, bookings, ledger = standing(*INITIAL), INITIAL, ()
attempts = []
for spec in REQUESTS:
    record = attempt(spec, commitment, bookings, ledger)
    attempts.append(record)
    commitment, bookings, ledger = record['result'].commitment, record['bookings_after'], record['result'].ledger

table(['Received', 'Client', 'Original feasible starts', 'Admitted duration', 'Admitted feasible starts', 'Outcome'],
      [(clock(a['received']), NAMES[a['request'].client], clock_interval(a['original_starts']),
        a['admission'].service.duration, clock_interval(a['admission'].feasible_starts), a['result'].status)
       for a in attempts])

| Received | Client | Original feasible starts | Admitted duration | Admitted feasible starts | Outcome |
| --- | --- | --- | --- | --- | --- |
| 08:05 | C | [09:00,09:30] | 60 | [09:00,09:30] | COMMITTED |
| 08:10 | D | ∅ | 30 | {10:00} | COMMITTED |
| 08:15 | E | [11:30,12:00] | 60 | [11:30,12:00] | NO-BLA |

## Timeline and bargaining rounds

- Four rows share one physical-time axis. Thin lines/dots are **start regions**; rectangles span **reserved duration**.
- Red shows D's rejected original start region and its earliest 60-minute window; the green/orange alternatives use the readmitted duration.
- E's outlines remain alternatives. Unreserved time after 11:30 remains free despite feasible offers extending to 13:00.

In [5]:
figure = plot_charging_morning(INITIAL, attempts, NAMES)
display_timeline(figure)

One charging point · three successive requests

In [6]:
table(['Client', 'Round', 'Proposer', 'Booking', 'Price (€)', 'Owner/client utilities', 'Accepted'],
      [(NAMES[a['request'].client], r.number, r.proposer, clock_interval(r.proposal.offer.interval),
        r.proposal.price, tuple(v for _, v in r.utilities), r.accepted)
       for a in attempts for r in a['negotiation'].history])
table(['After attempt', 'Earlier bookings unchanged', 'Jointly satisfiable', 'New agreement'],
      [(NAMES[a['request'].client], a['previous_unchanged'], a['jointly_satisfiable'],
        'None' if a['negotiation'].agreement is None else clock_interval(a['negotiation'].agreement.offer.interval))
       for a in attempts])

| Client | Round | Proposer | Booking | Price (€) | Owner/client utilities | Accepted |
| --- | --- | --- | --- | --- | --- | --- |
| C | 1 | owner | [09:00,10:00) | 6 | (Fraction(4, 1), Fraction(2, 1)) | False |
| C | 2 | client | [09:00,10:00) | 9/2 | (Fraction(5, 2), Fraction(7, 2)) | True |
| D | 1 | owner | [10:00,10:30) | 6 | (Fraction(4, 1), Fraction(2, 1)) | False |
| D | 2 | client | [10:00,10:30) | 9/2 | (Fraction(5, 2), Fraction(7, 2)) | True |
| E | 1 | owner | [11:30,12:30) | 8 | (Fraction(0, 1), Fraction(-2, 1)) | False |
| E | 2 | client | [11:30,12:30) | 6 | (Fraction(-2, 1), Fraction(0, 1)) | False |
| E | 3 | owner | [11:30,12:30) | 8 | (Fraction(0, 1), Fraction(-2, 1)) | False |

| After attempt | Earlier bookings unchanged | Jointly satisfiable | New agreement |
| --- | --- | --- | --- |
| C | True | True | [09:00,10:00) |
| D | True | True | [10:00,10:30) |
| E | True | True | None |

## Result checks

These assert the requested scenario outcomes **after** running the method; they do not select or force agreements.

In [7]:
C, D, E = attempts
assert tuple(o.interval for o in C['offers']) == (P.closedopen(540, 600), P.closedopen(570, 630))
assert C['admission'].service == C['request'].service
assert C['negotiation'].agreement.offer.interval == P.closedopen(540, 600)
assert D['original_starts'].empty and D['admission'].service == D['request'].alternatives[0]
assert tuple(o.interval for o in D['offers']) == (P.closedopen(600, 630),)
assert D['negotiation'].agreement.offer.interval == P.closedopen(600, 630)
assert tuple(o.interval for o in E['offers']) == (P.closedopen(690, 750), P.closedopen(720, 780))
assert E['result'].status == 'NO-BLA' and len(E['negotiation'].history) == ROUND_BUDGET
assert E['bookings_after'] == E['bookings_before'] and E['result'].ledger == D['result'].ledger
assert tuple((b.client, b.interval) for b in sorted(bookings, key=lambda b: b.start)) == (
    (1, P.closedopen(480, 540)), (3, P.closedopen(540, 600)),
    (4, P.closedopen(600, 630)), (2, P.closedopen(630, 690)))
print('Final schedule: 08:00–09:00 A → 09:00–10:00 C → 10:00–10:30 D → 10:30–11:30 B; then unreserved.')

Final schedule: 08:00–09:00 A → 09:00–10:00 C → 10:00–10:30 D → 10:30–11:30 B; then unreserved.


In [8]:
export_path = NOTEBOOK_DIR / 'charging_morning.html'
export_timelines(export_path, figure)
print(f'Generated {export_path}')

Generated notebooks/charging_morning.html
